# Hadoop_Ecommerce_Lab
Doanh thu theo category bằng Hadoop Streaming (Colab, local mode).
**Trước khi chạy:** upload 3 file `orders_2026_09.csv`, `mapper_revenue.py`, `reducer_revenue.py` vào `/content` (panel Files bên trái).

In [ ]:
!java -version
!ls -l /content

In [ ]:
!wget -q --show-progress https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz
!tar -xzf hadoop-3.5.0.tar.gz

In [ ]:
import os, subprocess
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["JAVA_HOME"] = os.path.realpath(subprocess.check_output(
    ["bash","-lc","dirname $(dirname $(readlink -f $(which java)))"], text=True).strip())
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]
with open(os.environ["HADOOP_HOME"] + "/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write("\nexport JAVA_HOME=" + os.environ["JAVA_HOME"] + "\n")
!hadoop version

In [ ]:
# Chạy thử pipeline trước khi gọi Hadoop
!cat /content/orders_2026_09.csv | python3 /content/mapper_revenue.py | sort | python3 /content/reducer_revenue.py

In [ ]:
import glob, shutil, subprocess
shutil.rmtree("/content/output_revenue", ignore_errors=True)
jar = glob.glob("/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming*.jar")[0]
subprocess.run(["hadoop","jar",jar,
  "-D","mapreduce.framework.name=local","-D","fs.defaultFS=file:///",
  "-files","/content/mapper_revenue.py,/content/reducer_revenue.py",
  "-mapper","python3 mapper_revenue.py","-reducer","python3 reducer_revenue.py",
  "-input","file:///content/orders_2026_09.csv","-output","file:///content/output_revenue"], check=True)

In [ ]:
!ls -l /content/output_revenue
!cat /content/output_revenue/part-*
!echo '--- TOP 3 ---'
!sort -k2,2nr /content/output_revenue/part-* | head -3

In [ ]:
from google.colab import files
files.download('/content/output_revenue/part-00000')